# 06 Minimum Margin と Batch-Wide Prediction Stability

## 今回の到達点

05 では、sample $n$ ごとに

$$
\|\delta l_n\|_\infty
\le
\|\Delta L\|_F
\le
\|X\|_F\|\Delta W_1\|_2\|W_2\|_2
$$

と、

$$
\|\delta l_n\|_\infty
<
\frac{\gamma_n}{2}
$$

を接続し、

$$
\|X\|_F\|\Delta W_1\|_2\|W_2\|_2
<
\frac{\gamma_n}{2}
$$

なら、sample $n$ の predicted class が変わらないことを確認しました。

今回は、batch 内の margin の最小値

$$
\boxed{
\gamma_{\min}
=
\min_{1\le n\le B}\gamma_n
}
$$

を使って、

$$
\boxed{
\|X\|_F\|\Delta W_1\|_2\|W_2\|_2
<
\frac{\gamma_{\min}}{2}
}
$$

なら、

$$
\boxed{
\operatorname*{argmax}_k(\widetilde l_n)_k
=
\operatorname*{argmax}_k(l_n)_k
\qquad
\text{for all }n
}
$$

すなわち、batch prediction vector 全体で

$$
\boxed{\widetilde y=y^\ast}
$$

が保証されることを数値的に確認します。

## 今回扱わないもの

- ground-truth label
- classification accuracy
- accuracy lower bound
- dataset 全体の保証
- minibatch をまたいだ global minimum margin
- TT-rank / bond dimension sweep
- rank と accuracy の比較
- fine-tuning
- training / backward

この Notebook は、04〜06で進めた prediction-stability certificate 理論の締めです。


## 1. 実行環境

数値的な不等式を安定して確認するため、今回も `float64` を使います。

既出の変数は05と同じ意味で使います。  
今回**新しく導入する変数**については、初出時に以下の6項目を説明します。

1. 何を意味する変数か
2. 数式上の定義
3. shape
4. 各軸の意味
5. 何から計算されるか
6. 何のために使うか


In [17]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
from torch.testing import assert_close

torch.manual_seed(0)

dtype = torch.float64
device = torch.device("cpu")

rtol = 1e-10
atol = 1e-12
eps = 1e-12


## 2. 05 と同じ toy MLP を用意する

ここは既習内容なので完成コードにします。

shape は、

$$
X\in\mathbb R^{B\times d},
\qquad
W_1,\widetilde W_1,\Delta W_1\in\mathbb R^{h\times d},
$$

$$
W_2\in\mathbb R^{c\times h},
\qquad
L,\widetilde L,\Delta L\in\mathbb R^{B\times c}.
$$

今回も、

$$
U
\coloneqq
\|X\|_F\|\Delta W_1\|_2\|W_2\|_2
$$

を 05 で得た共通上界として使います。

### 既出変数

| Python | 数式 | shape | 意味 |
|---|---|---:|---|
| `x` | $X$ | $(B,d)$ | batch input |
| `delta_w1` | $\Delta W_1$ | $(h,d)$ | 第1 Linear の weight approximation error |
| `logits` | $L$ | $(B,c)$ | dense logits |
| `logits_tilde` | $\widetilde L$ | $(B,c)$ | compressed logits |
| `delta_l` | $\Delta L$ | $(B,c)$ | batch logits error |
| `full_upper_bound` | $U$ | scalar | $\|\Delta L\|_F$ を上から抑える05の上界 |


In [18]:
B, d, h, c = 3, 2, 2, 3

x = torch.tensor(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ],
    dtype=dtype,
    device=device,
)

w1 = torch.tensor(
    [
        [2.0, 0.0],
        [0.0, 2.0],
    ],
    dtype=dtype,
    device=device,
)

b1 = torch.tensor(
    [1.0, 1.0],
    dtype=dtype,
    device=device,
)

delta_w1 = torch.tensor(
    [
        [0.01, -0.005],
        [0.004, 0.008],
    ],
    dtype=dtype,
    device=device,
)

w1_tilde = w1 + delta_w1

w2 = torch.tensor(
    [
        [2.0, 0.0],
        [0.0, 1.5],
        [-1.0, -1.0],
    ],
    dtype=dtype,
    device=device,
)

b2 = torch.zeros(c, dtype=dtype, device=device)

z1 = x @ w1.T + b1
h_dense = torch.relu(z1)
logits = h_dense @ w2.T + b2

z1_tilde = x @ w1_tilde.T + b1
h_tilde = torch.relu(z1_tilde)
logits_tilde = h_tilde @ w2.T + b2

delta_l = logits_tilde - logits

x_fro = torch.linalg.matrix_norm(x, ord="fro")
delta_w1_spectral = torch.linalg.matrix_norm(delta_w1, ord=2)
w2_spectral = torch.linalg.matrix_norm(w2, ord=2)

full_upper_bound = (
    x_fro
    * delta_w1_spectral
    * w2_spectral
)

delta_l_fro = torch.linalg.matrix_norm(delta_l, ord="fro")

assert logits.shape == (B, c)
assert logits_tilde.shape == (B, c)
assert delta_l.shape == (B, c)
assert full_upper_bound.shape == torch.Size([])
assert delta_l_fro <= full_upper_bound + eps


## 3. Batch 内の dense prediction vector を作る

各 sample の dense predicted class は、

$$
y_n^\ast
=
\operatorname*{argmax}_{k}(l_n)_k
$$

でした。

batch 全体では、

$$
y^\ast
=
\begin{bmatrix}
y_1^\ast\\
y_2^\ast\\
\vdots\\
y_B^\ast
\end{bmatrix}.
$$

### 新しい変数：`y_star`

1. **意味**: dense model の batch prediction vector
2. **数式上の定義**:
   $$
   y^\ast=(y_1^\ast,\ldots,y_B^\ast)^\mathsf T,
   \qquad
   y_n^\ast=\arg\max_k(l_n)_k
   $$
3. **shape**: $(B,)$
4. **各軸の意味**: axis 0 が sample index
5. **何から計算されるか**: dense logits $L$
6. **何のために使うか**: compressed prediction と比較する基準


In [19]:
# TODO 1:
from torch import argmax
# dense logits から、batch 全体の predicted class vector y_star を計算してください。

# その場だけ
print(logits.detach().cpu().numpy().round(4))

# 行ごとに
for n, row in enumerate(logits):
    print(f"n={n}: {row.tolist()}")
y_star = argmax(logits, dim=1)


[[ 6.   1.5 -4. ]
 [ 2.   4.5 -4. ]
 [ 6.   4.5 -6. ]]
n=0: [6.0, 1.5, -4.0]
n=1: [2.0, 4.5, -4.0]
n=2: [6.0, 4.5, -6.0]


### ヒント

`logits` の shape は `(B, c)` です。

各 row ごとに最大 logit の class index を取りたいので、`torch.argmax` の `dim` を考えてください。


### 検証


In [20]:
assert y_star.shape == (B,)
assert y_star.dtype == torch.int64

for n in range(B):
    top_value = logits[n, y_star[n]]
    assert torch.sum(logits[n] == top_value) == 1


## 4. Margin vector $\gamma$ を作る

各 sample の classification margin は、

$$
\gamma_n
=
(l_n)_{y_n^\ast}
-
\max_{k\neq y_n^\ast}(l_n)_k.
$$

batch 全体の margin を並べたものを、

$$
\boxed{
\gamma
=
\begin{bmatrix}
\gamma_1\\
\gamma_2\\
\vdots\\
\gamma_B
\end{bmatrix}
\in\mathbb R^B
}
$$

とします。

### 新しい変数：`top2_values`

1. **意味**: 各 sample の上位2つの dense logits
2. **数式上の定義**: 各 $l_n$ の largest / second-largest logit
3. **shape**: $(B,2)$
4. **各軸の意味**: axis 0 = sample、axis 1 = 1位 / 2位
5. **何から計算されるか**: dense logits $L$
6. **何のために使うか**: $\gamma_n=$ 1位 − 2位 を計算する

### 新しい変数：`gamma`

1. **意味**: batch 内の全 sample の classification margin vector
2. **数式上の定義**:
   $$
   \gamma=(\gamma_1,\ldots,\gamma_B)^\mathsf T
   $$
3. **shape**: $(B,)$
4. **各軸の意味**: axis 0 = sample index
5. **何から計算されるか**: 各 sample の top-1 / top-2 dense logit
6. **何のために使うか**: batch 内で最も厳しい margin を求める


In [21]:
# TODO 2:
# 各 sample の top-2 logits と margin vector gamma を計算してください。

# 各行（sample）について、class 方向の上位2つの logit を取る。
# top2_values[:, 0] = 1位, top2_values[:, 1] = 2位
# shape: (B, 2)
top2_values = torch.topk(logits, k=2, dim=1).values

# classification margin: γ_n = 1位 − 2位
# shape: (B,)
gamma = top2_values[:, 0] - top2_values[:, 1]


### ヒント

各 sample の margin は、

$$
\text{top-1 logit}
-
\text{top-2 logit}
$$

です。

`torch.topk(..., k=2, dim=...)` が使えます。


### 検証


In [22]:
assert top2_values.shape == (B, 2)
assert gamma.shape == (B,)
assert torch.all(gamma > 0.0)

assert_close(
    gamma,
    top2_values[:, 0] - top2_values[:, 1],
    rtol=rtol,
    atol=atol,
)


## 5. Minimum margin $\gamma_{\min}$

minimum margin を、

$$
\boxed{
\gamma_{\min}
=
\min_{1\le n\le B}\gamma_n
}
$$

と定義します。

これは、batch 内で最も margin の小さい sample の margin です。

重要なのは、minimum-margin sample は **margin に基づく保証で最も厳しい sample** という意味であり、実際の圧縮で必ず最初に prediction が変わる sample という意味ではないことです。

### 新しい変数：`gamma_min`

1. **意味**: batch 内の最小 classification margin
2. **数式上の定義**: $\gamma_{\min}=\min_n\gamma_n$
3. **shape**: scalar tensor、`torch.Size([])`
4. **各軸の意味**: scalar なので軸なし
5. **何から計算されるか**: margin vector $\gamma$
6. **何のために使うか**: 全 sample に共通する最も厳しい threshold を作る

### 新しい変数：`bottleneck_index`

1. **意味**: minimum margin を持つ sample index
2. **数式上の定義**: $n_{\mathrm{bottleneck}}=\operatorname*{argmin}_n\gamma_n$
3. **shape**: scalar integer tensor
4. **各軸の意味**: 軸なし。値が sample index
5. **何から計算されるか**: margin vector $\gamma$
6. **何のために使うか**: batch certificate を最も厳しくする sample を確認する


In [25]:
# TODO 3:
# minimum margin と bottleneck sample index を計算してください。
from torch import argmin
gamma_min = torch.min(gamma)
bottleneck_index = argmin(gamma)
# gamma_min, bottleneck_index = torch.min(gamma, dim=0)


### ヒント

必要なのは `gamma` の minimum と、その位置です。


### 検証


In [26]:
assert gamma_min.shape == torch.Size([])
assert bottleneck_index.shape == torch.Size([])
assert bottleneck_index.dtype == torch.int64

assert_close(
    gamma_min,
    gamma[bottleneck_index],
    rtol=rtol,
    atol=atol,
)

assert torch.all(gamma_min <= gamma + eps)


## 6. なぜ $\gamma_{\min}/2$ を使えば全 sample に効くのか

minimum の定義から、

$$
\gamma_{\min}\le\gamma_n
\qquad
\text{for all }n.
$$

$2>0$ なので、両辺を2で割っても不等号の向きは変わりません。

$$
\boxed{
\frac{\gamma_{\min}}{2}
\le
\frac{\gamma_n}{2}
\qquad
\text{for all }n
}
$$

したがって、$\gamma_{\min}/2$ は batch 内で最も小さい sample-wise threshold です。

### 新しい変数：`minimum_threshold_is_smallest`

1. **意味**: $\gamma_{\min}/2$ が全 sample の $\gamma_n/2$ 以下か
2. **数式上の定義**: $\forall n,\ \gamma_{\min}/2\le\gamma_n/2$
3. **shape**: Python bool または scalar bool
4. **各軸の意味**: 軸なし
5. **何から計算されるか**: `gamma_min`, `gamma`
6. **何のために使うか**: batch-wide condition が全 sample に伝わる根拠を確認する


In [34]:
# TODO 4:
# gamma_min / 2 <= gamma_n / 2 が全 sample で成立することを確認してください。
minimum_threshold_is_smallest = torch.all(gamma_min / 2 <= gamma / 2)


### ヒント

vector 比較の結果を1個の真偽値にまとめるには `torch.all` が使えます。


### 検証


In [35]:
assert bool(minimum_threshold_is_smallest)


## 7. 共通上界 $U$ と minimum margin を接続する

05で、

$$
U
\coloneqq
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
$$

と置きました。

任意の sample $n$ について、

$$
\|\delta l_n\|_\infty\le U
$$

です。

もし、

$$
\boxed{
U<\frac{\gamma_{\min}}{2}
}
$$

なら、

$$
\|\delta l_n\|_\infty
\le
U
<
\frac{\gamma_{\min}}{2}
\le
\frac{\gamma_n}{2}.
$$

したがって、

$$
\boxed{
\|\delta l_n\|_\infty
<
\frac{\gamma_n}{2}
}
$$

が **全 sample で同時に成立**します。

### 新しい変数：`batch_certificate`

1. **意味**: batch 全体の prediction stability を保証する十分条件が成立したか
2. **数式上の定義**: $U<\gamma_{\min}/2$
3. **shape**: scalar bool
4. **各軸の意味**: 軸なし
5. **何から計算されるか**: `full_upper_bound`, `gamma_min`
6. **何のために使うか**: 全 sample の sample-wise certificate を一括で保証する


### TODO 5 の前に固定 toy setting の数値を確認する

今回は certificate が成立する設定を使っています。

固定値を変更して certificate が成立しなくなっても、理論が間違ったのではなく、**この sufficient condition では batch 全体を保証できない**という意味です。


In [36]:
print(f"||Delta L||_F       = {delta_l_fro.item():.8f}")
print(f"U                    = {full_upper_bound.item():.8f}")
print("gamma                =", gamma.tolist())
print(f"gamma_min             = {gamma_min.item():.8f}")
print(f"gamma_min / 2         = {(gamma_min / 2).item():.8f}")
print(f"bottleneck sample idx = {int(bottleneck_index)}")


||Delta L||_F       = 0.03997499
U                    = 0.05221958
gamma                = [4.5, 2.5, 1.5]
gamma_min             = 1.50000000
gamma_min / 2         = 0.75000000
bottleneck sample idx = 2


In [45]:
# TODO 5:
# batch-wide sufficient condition U < gamma_min / 2 を判定してください。
batch_certificate = torch.all(full_upper_bound < gamma_min / 2)


### ヒント

ここでの $U$ は `full_upper_bound` です。


### 検証


In [ ]:
# この toy setting では certificate が成立するように
# delta_w1 を十分小さく選んでいます。
assert bool(batch_certificate)
assert full_upper_bound < gamma_min / 2


: 

## 8. Batch certificate から全 sample の条件へ

batch certificate が成立すると、

$$
U<\frac{\gamma_{\min}}{2}
\le
\frac{\gamma_n}{2}
$$

なので、すべての sample $n$ について、

$$
U<\frac{\gamma_n}{2}
$$

です。

### 新しい変数：`sample_certificates`

1. **意味**: 各 sample に対し、共通上界 $U$ が sample-wise threshold より小さいか
2. **数式上の定義**: $[U<\gamma_1/2,\ldots,U<\gamma_B/2]$
3. **shape**: $(B,)$ の bool tensor
4. **各軸の意味**: axis 0 = sample index
5. **何から計算されるか**: $U$ と margin vector $\gamma$
6. **何のために使うか**: batch certificate が各 sample の certificate を同時に満たすことを確認する


In [41]:
# TODO 6:
# 各 sample の U < gamma_n / 2 をまとめて判定してください。
sample_certificates = full_upper_bound  < gamma / 2


### ヒント

`gamma / 2` は shape `(B,)` です。  
scalar `full_upper_bound` との比較は broadcasting されます。


### 検証


In [42]:
assert sample_certificates.shape == (B,)
assert sample_certificates.dtype == torch.bool
assert torch.all(sample_certificates)


## 9. Dense / Compressed の prediction vector を比較する

compressed model の prediction vector を、

$$
\widetilde y
=
\begin{bmatrix}
\widetilde y_1\\
\widetilde y_2\\
\vdots\\
\widetilde y_B
\end{bmatrix},
\qquad
\widetilde y_n
=
\operatorname*{argmax}_k(\widetilde l_n)_k
$$

とします。

### 新しい変数：`y_tilde`

1. **意味**: compressed model の batch prediction vector
2. **数式上の定義**: $\widetilde y=(\widetilde y_1,\ldots,\widetilde y_B)^\mathsf T$
3. **shape**: $(B,)$
4. **各軸の意味**: axis 0 = sample index
5. **何から計算されるか**: compressed logits $\widetilde L$
6. **何のために使うか**: dense predictions $y^\ast$ と比較する

### 新しい変数：`all_predictions_match`

1. **意味**: batch 内の全 prediction が dense / compressed で一致したか
2. **数式上の定義**: $\widetilde y=y^\ast$
3. **shape**: scalar bool
4. **各軸の意味**: 軸なし
5. **何から計算されるか**: `y_tilde`, `y_star`
6. **何のために使うか**: certificate が保証する結論を実測値でも確認する


In [43]:
# TODO 7:
# compressed prediction vector と、全 prediction が一致するかを計算してください。
y_tilde = torch.argmax(logits_tilde, dim=1)
all_predictions_match = torch.all(y_tilde == y_star)


### ヒント

- `y_tilde` は `logits_tilde` から sample ごとに argmax を取ります。
- vector 全体の一致は、要素ごとの比較を `torch.all` でまとめられます。


### 検証


In [44]:
assert y_tilde.shape == (B,)
assert y_tilde.dtype == torch.int64
assert bool(all_predictions_match)
assert torch.equal(y_tilde, y_star)


## 10. 最終 chain

今回確認した流れは、

$$
\|\delta l_n\|_\infty
\le
U
$$

$$
U
<
\frac{\gamma_{\min}}{2}
$$

$$
\frac{\gamma_{\min}}{2}
\le
\frac{\gamma_n}{2}
\qquad
\text{for all }n
$$

なので、

$$
\boxed{
\|\delta l_n\|_\infty
\le
U
<
\frac{\gamma_{\min}}{2}
\le
\frac{\gamma_n}{2}
\qquad
\text{for all }n
}
$$

です。

したがって、04の sample-wise stability condition を全 sample に適用でき、

$$
\boxed{
U
<
\frac{\gamma_{\min}}{2}
\quad\Longrightarrow\quad
\widetilde y=y^\ast
}
$$

を得ます。

$U$ を元の式に戻すと、

$$
\boxed{
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
<
\frac{\gamma_{\min}}{2}
\quad\Longrightarrow\quad
\widetilde y=y^\ast
}
$$

です。


## 11. この条件は十分条件であって必要条件ではない

batch certificate

$$
U<\frac{\gamma_{\min}}{2}
$$

が成立しなかった場合に言えるのは、**この certificate では batch 全体の prediction stability を保証できない**ということだけです。

「batch 内で prediction change が必ず起きる」とは言えません。

理由は、

- $U$ 自体が actual logits error よりかなり大きい可能性がある
- sample ごとの actual error はさらに小さい可能性がある
- error direction が argmax を壊す方向とは限らない
- bottleneck sample でも実際には順位を保つ方向へ logits が動く場合がある

ためです。

これは04・05で確認した「certificate と実測 prediction は別物」という考え方の batch-wide 版です。


## 12. まとめ

今回新しく加わったのは、

$$
\boxed{
\gamma_{\min}
=
\min_n\gamma_n
}
$$

です。

minimum margin を使うことで、

$$
\gamma_{\min}
\le
\gamma_n
\qquad
\text{for all }n
$$

から、

$$
\frac{\gamma_{\min}}{2}
\le
\frac{\gamma_n}{2}
\qquad
\text{for all }n
$$

を得ます。

したがって、

$$
\boxed{
\|X\|_F
\|\Delta W_1\|_2
\|W_2\|_2
<
\frac{\gamma_{\min}}{2}
}
$$

なら、batch 内の全 sample で、

$$
\operatorname*{argmax}_k(\widetilde l_n)_k
=
\operatorname*{argmax}_k(l_n)_k
$$

が同時に成立し、

$$
\boxed{\widetilde y=y^\ast}
$$

が保証されます。

ここで保証しているのは **dense / compressed prediction agreement** であり、ground-truth に対する correctness や accuracy ではありません。

これで04〜06の prediction-stability certificate 理論はいったん閉じます。

次は学習スケジュール本筋へ戻り、

$$
\boxed{\text{TT-rank / bond dimension sweep}}
$$

へ進みます。
